In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import pickle
import os
from torch_geometric.datasets import BA2MotifDataset
from torch.utils.data import Subset

In [6]:
class GINGraphClf(nn.Module):
    def __init__(self, in_dim,out_dim, hidden_dim=64):
        super().__init__()
        self.conv1=GINConv(nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        self.conv2=GINConv(nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        self.conv3=GINConv(nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        
        self.mlp = nn.Sequential(
            nn.Linear(3*hidden_dim, hidden_dim//2),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, out_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x, edge_index, batch):
        h1=self.conv1(x, edge_index)
        h2=self.conv2(h1, edge_index)
        h3=self.conv3(h2, edge_index)

        h1=global_add_pool(h1,batch)
        h2=global_add_pool(h2,batch)
        h3=global_add_pool(h3,batch)    


        x=torch.cat([h1,h2,h3],dim=1)


        return self.mlp(x)

In [7]:
dataset_path='./data/BA2Motif'
with open('./data/splits.pkl', 'rb') as f:
    splits=pickle.load(f)

full_dataset=BA2MotifDataset(root=dataset_path)
test_dataset=Subset(full_dataset,splits['test'])


test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [8]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=GINGraphClf(test_dataset[0].num_features,2).to(device)


checkpoint=torch.load('./models/GIN/best_gin_model.pt',map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [9]:
model.eval()
for param in model.parameters():
    param.requires_grad = False

print(f"Model loaded from './models/best_gcn_model.pt'")
print(f"Best validation accuracy during training: {checkpoint['best_val_acc']:.4f}")

Model loaded from './models/best_gcn_model.pt'
Best validation accuracy during training: 1.0000


In [10]:
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            pred = out.argmax(dim=1)
            correct += (pred == data.y).sum().item()
            total += data.num_graphs
    return correct / total

test_acc = evaluate(model, test_loader, device)
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 1.0000


In [11]:
correct_indices = []
with torch.no_grad():
    for idx, data in enumerate(test_dataset):
        data = data.to(device)
        batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
        out = model(data.x, data.edge_index, batch)
        pred = out.argmax(dim=1).item()
        if pred == data.y.item():
            correct_indices.append(idx)

print(f"Number of correctly classified test graphs: {len(correct_indices)} / {len(test_dataset)}")

Number of correctly classified test graphs: 100 / 100
